# SN-04 — Sirenisation, phase 3 : matching approfondi

Reprend les personnes morales non resolues en phase 2 — statut douteux, rejete
ou sans candidat — et ajoute deux criteres au scoring : le code APE et l'annee
d'ouverture.

Ces deux informations n'existent pas dans FINESS+ : elles proviennent des
extractions `Extraction_PM_codeape*.csv et Extraction_PM_datecreat*.csv`
deposees dans `data/referentiels/`. Chaque fichier porte le numero FINESS puis la
valeur ; les deux sont fusionnes sur le numero FINESS.

Un fichier manquant n'est pas bloquant : le critere correspondant est neutralise
et la ponderation s'adapte a ce qui reste disponible.

Ponderation adaptative, inchangee : 0,70 global + 0,20 APE + 0,10 date quand les
deux sont disponibles, 0,80 + 0,20 quand un seul l'est, score global seul sinon.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.sirenisation import CONFIG_SN, COLS_UL_EXPORT
from src.phases import executer_phase
from src.referentiel import charger_referentiel, enrichir
from src.scoring import extraire_annee
from src.excel_export import export_topn_excel, LABELS
from src.display import afficher_tableau, afficher_synthese
from config.settings import (
    SN_PERIMETRE, SIRENE_UL_CLEAN, SIRENE_UL_PARTITIONNE,
    SN_VALIDES_P1, SN_PHASE2, SN_PHASE3, RESULTS_SN_DIR, BUDGET_MEMOIRE_GO,
)

RESULTS_SN_DIR.mkdir(parents=True, exist_ok=True)
NON_RESOLUS = {"DOUTEUX", "REJETE", "SANS_CANDIDAT"}

sirens_valides_p1 = set(
    pd.read_parquet(SN_VALIDES_P1)["siren_norm_pm"].dropna().astype(str))
sirens_valides_p1.discard("")

In [2]:
df_p2 = pd.read_excel(SN_PHASE2, sheet_name="Top5", dtype=str)
df_p2["rang"] = pd.to_numeric(df_p2["rang"], errors="coerce")
rang1 = df_p2[df_p2["rang"] == 1]
ids_phase3 = set(rang1[rang1["statut_candidat"].isin(NON_RESOLUS)]["NB_PmSmsseId"]
                 .dropna().astype(str))

df_perimetre = pd.read_parquet(SN_PERIMETRE)
df_phase3 = df_perimetre[
    df_perimetre["NB_PmSmsseId"].astype(str).isin(ids_phase3)
].copy().reset_index(drop=True)
print(f"PM a traiter en phase 3 : {len(df_phase3):,}")

PM a traiter en phase 3 : 3,476


In [3]:
from src.referentiel import diagnostiquer
display(diagnostiquer())

,niveau,type,fichier,lignes,renseignes,colonnes,exemples
0,pm,ape,Extraction_PM_codeape.csv,98158,31170,nofiness | codeape,010000016=8710A | 010000156=8610Z | 010000164=...
1,pm,date,Extraction_PM_datecreat.csv,98158,98158,nofiness | dateouvert,010000016=01/01/1994 | 010000156=01/01/2001 | ...
2,ege,ape,Extraction_EG_codeape.csv,174438,67292,nofiness | codeape,010000024=8610Z | 010000032=8610Z | 010000040=...
3,ege,date,Extraction_EG_dateouverture_20260728.csv,174438,170225,nofiness | dateouvert,010000024=1979-02-13 | 010000032=1901-01-01 | ...


In [4]:
referentiel_pm = charger_referentiel("pm")
df_phase3 = enrichir(df_phase3, "finess_norm_pm", "pm", referentiel_pm)

print()
print(f"APE renseigne  : {(df_phase3['ape_referentiel'] != '').sum():,} / {len(df_phase3):,}")
print(f"Annee renseignee: {df_phase3['annee_referentiel'].notna().sum():,} / {len(df_phase3):,}")

Referentiel PM
   ape   Extraction_PM_codeape.csv : 98,158 lignes, 31,170 exploitables
   date  evenements_ouverture_pm.parquet : 98,194 PM
   total 98,194 structures, 31,170 avec APE, 98,194 avec annee
Enrichissement PM sur 3,476 lignes : APE 33.6 %, annee 100.0 %

APE renseigne  : 1,169 / 3,476
Annee renseignee: 3,476 / 3,476


In [5]:
def preparer_bloc_ul(df):
    df = df.copy()
    if "dateCreationUniteLegale" in df.columns:
        df["annee_creation_ul"] = df["dateCreationUniteLegale"].apply(extraire_annee)
    return df


In [6]:
df_top3 = executer_phase(
    df_phase3, CONFIG_SN,
    chemin_partitionne=SIRENE_UL_PARTITIONNE,
    chemin_complet=SIRENE_UL_CLEAN,
    n=3, approfondi=True,
    identifiants_deja_valides=sirens_valides_p1,
    colonnes_sirene=COLS_UL_EXPORT + ["annee_creation_ul"],
    budget_go=BUDGET_MEMOIRE_GO,
    preparer_bloc=preparer_bloc_ul,
    niveau_matching="departement",
)

df_top3 = df_top3.rename(columns={"siren": "siren_ref_app",
                                  "nom_sirene_retenu": "nom_ul_retenu",
                                  "identifiant_coherent": "siren_coherent",
                                  "score_ajuste": "score_approfondi_ajuste"})

rang1 = df_top3[df_top3["rang"] == 1]
print(f"\nLignes top 3 : {len(df_top3):,}")
print(f"   avec score APE  : {df_top3['score_ape'].notna().sum():,}")
print(f"   avec score date : {df_top3['score_date'].notna().sum():,}")
print()
print(rang1["statut_candidat"].value_counts())

Phase 3 (departement):   0%|          | 0/101 [00:00<?, ?it/s]


Lignes top 3 : 10,428
   avec score APE  : 3,507
   avec score date : 10,428

statut_candidat
DOUTEUX        2334
VALIDE         1109
REJETE           30
VALIDE_FORT       3
Name: count, dtype: int64


In [7]:
afficher_tableau(
    df_top3[["NB_PmSmsseId", "TX_DenominationPm", "siren_ref_app",
             "denominationUniteLegale", "nom_ul_retenu",
             "ape_referentiel", "activitePrincipaleUniteLegale", "score_ape",
             "annee_referentiel", "annee_creation_ul", "score_date",
             "score_global", "score_global_approfondi",
             "statut_candidat", "rang"]],
    "Apercu top 3 phase 3", max_lignes=10)

NB_PmSmsseId,TX_DenominationPm,siren_ref_app,denominationUniteLegale,nom_ul_retenu,ape_referentiel,activitePrincipaleUniteLegale,score_ape,annee_referentiel,annee_creation_ul,score_date,score_global,score_global_approfondi,statut_candidat,rang
25,EHPAD ST TRIVIER DE COURTES,260100243,MAISON DE RETRAITE ST TRIVIER DE COURTES,MAISON RETRAITE ST TRIVIER COURTES,8730A,87.30A,100.000000,2001,1983,20.000000,48.860000,56.200000,VALIDE,1
25,EHPAD ST TRIVIER DE COURTES,441504818,DE L'ETANG FOISSIAT,ETANG FOISSIAT,8730A,01.41Z,0.000000,2001,2002,100.000000,51.880000,46.320000,VALIDE,2
25,EHPAD ST TRIVIER DE COURTES,751371899,COMITE FOIRE EXPOSITION ST TRIVIER DE COURTES,COMITE FOIRE EXPOSITION ST TRIVIER COURTES,8730A,94.99Z,0.000000,2001,2001,100.000000,48.690000,44.080000,DOUTEUX,3
171,PHARMACIE LETIERCE-LEHMANN,902691385,CS BOURG EN BRESSE,CS BOURG BRESSE,,56.10C,nan,1942,2021,0.000000,63.510000,50.810000,DOUTEUX,1
171,PHARMACIE LETIERCE-LEHMANN,880749452,ASSOCIATION DES PROFS DE CONDUITE ROUTIERE DU LYCEE GABRIEL VOISIN,PROFS CONDUITE ROUTIERE LYCEE GABRIEL VOISIN,,94.99Z,nan,1942,2018,0.000000,61.430000,49.150000,VALIDE,2
171,PHARMACIE LETIERCE-LEHMANN,779310317,L AVANT GARDE DU LYCEE LALANDE,AVANT GARDE LYCEE LALANDE,,93.12Z,nan,1942,1945,80.000000,39.860000,47.890000,DOUTEUX,3
191,PHARMACIE BOUVIER,402225759,None,,,47.73Z,nan,1995,1995,100.000000,22.610000,38.090000,DOUTEUX,1
191,PHARMACIE BOUVIER,388876203,BOURNONVILLE,BOURNONVILLE,,43.22B,nan,1995,1992,80.000000,40.550000,48.440000,DOUTEUX,2
191,PHARMACIE BOUVIER,411166614,SARL PORTITAL,PORTITAL,,43.33Z,nan,1995,1997,80.000000,38.050000,46.440000,DOUTEUX,3
213,PHARMACIE BENDAHOU,881892442,AMICALE BOULE DE FOISSIAT,AMICALE BOULE FOISSIAT,,93.12Z,nan,1972,1971,100.000000,36.160000,48.930000,DOUTEUX,1


In [8]:
COLS_EXPORT = [
    "ID_PmSmsse", "NB_PmSmsseId", "TX_NumFinessPm", "TX_Siren",
    "TX_CategorieEntiteGeographiqueExercice", "TX_DenominationPm",
    "cdcommune_norm_pm", "adresse_complete_pm", "sous_ensemble",
    "ape_referentiel", "annee_referentiel",
    "siren_ref_app", "siren_coherent", "bonus_applique",
    "denominationUniteLegale", "sigleUniteLegale", "nom_ul_retenu",
    "adresse_siege_complete_ul", "codeCommuneEtablissement",
    "categorieJuridiqueUniteLegale", "activitePrincipaleUniteLegale",
    "dateCreationUniteLegale",
    "score_nom", "score_adresse", "score_global",
    "annee_creation_ul", "score_ape", "score_date", "score_global_approfondi",
    "score_approfondi_ajuste", "statut_candidat", "rang",
]

df_top3["rang"] = df_top3["rang"].astype(int)
compteurs = export_topn_excel(df_top3, SN_PHASE3, COLS_EXPORT, sheet_name="Top3")

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  "Synthese phase 3 sirenisation (rang 1)")
print(f"\nFichier : {SN_PHASE3}")

Statut,Nb,% du total
Valide_fort,3,0.1%
Valide,"1,109",31.9%
Douteux,"2,334",67.1%
Rejeté,30,0.9%
Sans_candidat,0,0.0%
TOTAL,"3,476",100.0%



Fichier : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/sirenisation/sirenisation_phase3_approfondi.xlsx
